# Phân loại cảm xúc văn bản

Notebook khởi đầu của ML Academy VN. **Các kết quả bên dưới là kết quả thật khi chạy bằng Python 3 + scikit-learn lúc phát triển** (số có thể lệch nhẹ giữa các phiên bản thư viện). Hãy chạy lại, chỉnh sửa và hoàn thành các bước:

1. Chạy notebook trên tập 40 câu mẫu: bạn sẽ thấy kết quả gần mức ngẫu nhiên vì tập quá nhỏ. Đó chính là bài học về nhu cầu dữ liệu.
2. Tìm một tập dữ liệu cảm xúc tiếng Việt công khai, ghi lại nguồn và giấy phép.
3. Vector hóa bằng TF-IDF (thử n-gram 1-2) và huấn luyện Logistic Regression.
4. Đánh giá bằng cross-validation và F1 macro trên dữ liệu thật.
5. Đọc các câu dự đoán sai (phủ định, mỉa mai) và đề xuất cải tiến.

Baseline: Dự đoán lớp phổ biến nhất · Chỉ số: F1 macro

In [1]:
# Tập mẫu 40 câu do dự án tự biên soạn (không phải dữ liệu thật; chỉ để thực hành quy trình)
pos = ["phim rất hay và cảm động", "diễn viên diễn xuất xuất sắc", "nội dung hấp dẫn từ đầu đến cuối", "tôi rất thích bộ phim này", "kịch bản chặt chẽ và bất ngờ", "hình ảnh đẹp âm nhạc tuyệt vời", "một trải nghiệm đáng xem", "cả nhà đều khen hay", "cái kết làm tôi xúc động", "đạo diễn làm rất tốt", "phim vui nhộn xem rất đã", "nhân vật được xây dựng sâu sắc", "âm thanh sống động hình ảnh đẹp", "đáng tiền vé tôi sẽ xem lại", "diễn biến cuốn hút khó rời mắt", "phim ý nghĩa và nhân văn", "dàn diễn viên ăn ý tự nhiên", "phần hai còn hay hơn phần một", "tôi rất hài lòng với bộ phim", "tuyệt vời không có gì để chê"]
neg = ["phim rất dở và nhàm chán", "diễn viên diễn xuất tệ", "nội dung lê thê khó theo dõi", "tôi không thích bộ phim này", "kịch bản rời rạc vô lý", "hình ảnh xấu âm thanh rè", "một trải nghiệm đáng thất vọng", "cả nhà đều chê dở", "cái kết làm tôi tức giận", "đạo diễn làm quá tệ", "phim buồn ngủ xem rất chán", "nhân vật nhạt nhòa không cảm xúc", "âm thanh lỗi hình ảnh mờ", "phí tiền vé tôi sẽ không xem lại", "diễn biến chậm chạp dễ ngủ gật", "phim vô nghĩa và nhảm nhí", "dàn diễn viên gượng gạo thiếu tự nhiên", "phần hai tệ hơn phần một", "tôi rất thất vọng với bộ phim", "quá tệ không có gì để khen"]
texts = pos + neg; y = [1] * len(pos) + [0] * len(neg)
print(len(texts), 'câu;', sum(y), 'tích cực')

40 câu; 20 tích cực


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, clf in {'baseline': DummyClassifier(strategy='most_frequent'), 'TF-IDF 1-gram + LogReg': LogisticRegression(max_iter=1000)}.items():
    s = cross_val_score(make_pipeline(TfidfVectorizer(), clf), texts, y, cv=cv, scoring='f1')
    print(f'{name:24s} F1(CV)={s.mean():.3f} ± {s.std():.3f}')
s = cross_val_score(make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000)), texts, y, cv=cv, scoring='f1')
print(f'{"TF-IDF 1-2gram + LogReg":24s} F1(CV)={s.mean():.3f} ± {s.std():.3f}')

baseline                 F1(CV)=0.000 ± 0.000
TF-IDF 1-gram + LogReg   F1(CV)=0.268 ± 0.174
TF-IDF 1-2gram + LogReg  F1(CV)=0.211 ± 0.203


In [3]:
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000)).fit(texts, y)
new = ['phim này không hay', 'diễn xuất tuyệt vời', 'nội dung nhảm nhí', 'tôi rất thích']
for t, p in zip(new, model.predict_proba(new)[:, 1]): print(f'{t:24s} P(tích cực)={p:.2f}')
print('Nhận xét: với chỉ 40 câu, mỗi câu mang nhiều từ riêng nên mô hình thấy rất ít từ dùng chung giữa train và test; F1 thấp và xác suất gần 0.5 nghĩa là gần như đoán mò. Cần nhiều dữ liệu hơn nhiều.')

phim này không hay       P(tích cực)=0.51
diễn xuất tuyệt vời      P(tích cực)=0.58
nội dung nhảm nhí        P(tích cực)=0.45
tôi rất thích            P(tích cực)=0.56
Nhận xét: với chỉ 40 câu, mỗi câu mang nhiều từ riêng nên mô hình thấy rất ít từ dùng chung giữa train và test; F1 thấp và xác suất gần 0.5 nghĩa là gần như đoán mò. Cần nhiều dữ liệu hơn nhiều.


## Việc của bạn

- Thử thêm một mô hình hoặc siêu tham số khác và so sánh công bằng bằng cross-validation.
- Viết 3-5 câu nhận xét: chênh lệch nhỏ giữa các mô hình có đáng tin không?